## 0. Prérequis

1. **Exécution → Modifier le type d'exécution → T4 GPU**.
2. Vérifier que le GPU est bien alloué (cellule suivante).
3. Compter **moins de 10 minutes** par entraînement.

> Toute expérience part de `src/config.py` et ne modifie **qu'un seul**
> hyperparamètre par rapport à la baseline.


In [ ]:
!nvidia-smi

## 1. Cloner le dépôt et installer les dépendances

In [ ]:
import os

REPO_URL = "https://github.com/jaouheraya12-create/transformer-hyperparam-study.git"
REPO_DIR = "transformer-hyperparam-study"

# Idempotent : on clone seulement si le dossier n'existe pas déjà.
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}

%cd {REPO_DIR}
!git pull --ff-only


In [ ]:
!pip install -q -r requirements.txt

## 2. Vérifier la configuration

On vérifie que le paquet `src` est importable et que la configuration de
référence est bien celle validée par l'équipe.


In [ ]:
!python -c "import src.config; print('config OK')"

from src.config import Config

cfg = Config()
print(cfg.resume())
print("Splits :", cfg.train_size, "/", cfg.val_size, "/", cfg.test_size)


# 04 — Protocole d'évaluation, interactions et ablation (Oussama)

**Expériences 13 à 15** + **protocole d'évaluation** commun + **analyse des erreurs**.

## Protocole d'évaluation (commun à toute l'équipe)

| Élément | Choix |
|---|---|
| Jeu de test | 1 000 exemples du split officiel IMDB, jamais vus à l'entraînement |
| Métrique principale | **F1 binaire** (classe positive = 1) |
| Métriques secondaires | accuracy, precision, recall |
| Sélection du modèle | meilleure époque selon le **F1 de validation** |
| Rapport | `classification_report.txt` + matrice de confusion |
| Reproductibilité | seed 42, splits identiques, `run_config.json` par run |

> Le test n'est utilisé **qu'une fois par expérience**, à la fin. Ne jamais
> régler les hyperparamètres sur le test : c'est la validation qui sert à ça.


## Expérience 13 — interaction

Combine les meilleurs réglages individuels : `lr=5e-5`, `warmup=0.1`,
`cosine`, `batch=64`. C'est la seule expérience (avec la 15) qui modifie
plusieurs hyperparamètres à la fois — son but est de vérifier que les effets
se cumulent.


In [ ]:
!python -m src.train --id 13 --owner Oussama --name interaction_lr5e-5_warmup0.1_cosine_batch64 \
    --lr 5e-5 --warmup 0.1 --scheduler cosine --batch-size 64


## Expérience 14 — ablation

Ni warmup ni décroissance : `scheduler=constant`, `warmup=0.0`.
À comparer à l'exp. 1 (baseline, linear sans warmup) pour isoler l'apport réel
du scheduler.


In [ ]:
!python -m src.train --id 14 --owner Oussama --name ablation_constant_sans_warmup --scheduler constant


## Expérience 15 — meilleure combinaison (à définir)

À définir **après** analyse des expériences 1 à 14. Remplacer les valeurs
ci-dessous par la combinaison retenue et justifier le choix dans la colonne
`observations` du plan.


In [ ]:
# Exemple à adapter après analyse :
# !python -m src.train --id 15 --owner Oussama --name meilleure_combinaison \
#     --lr 5e-5 --warmup 0.1 --scheduler cosine --batch-size 64


## Consolidation des 15 expériences

In [ ]:
import pandas as pd

df = pd.read_csv("results/all_experiments.csv")
df = df.sort_values("id")
print(f"{len(df)} runs enregistrés")
df[["id", "owner", "name", "learning_rate", "warmup_ratio",
    "scheduler", "batch_size", "optimizer", "train_time_s", "f1"]]


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(df["id"].astype(str), df["f1"])
axes[0].set_xlabel("Expérience")
axes[0].set_ylabel("F1 (test)")
axes[0].set_title("F1 par expérience")
axes[0].tick_params(axis="x", labelsize=8)

axes[1].scatter(df["train_time_s"], df["f1"])
axes[1].set_xlabel("Temps d'entraînement (s)")
axes[1].set_ylabel("F1 (test)")
axes[1].set_title("Compromis temps / qualité")

plt.tight_layout()
plt.savefig("results/oussama/figures/synthese_15_experiences.png", dpi=150)
plt.show()


## Analyse des erreurs (faux positifs / faux négatifs)

Nécessite un modèle sauvegardé sur Google Drive (voir notebook 01).
Adapter `MODELE_DIR` au run analysé.


In [ ]:
# Décommenter après avoir monté Drive et sauvegardé un modèle.
# import torch
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# from src.config import Config
# from src.data import build_datasets
#
# MODELE_DIR = "/content/drive/MyDrive/hyperparam-study/models/exp_15_meilleure_combinaison"
# cfg = Config()
# dataset, _ = build_datasets(cfg, AutoTokenizer.from_pretrained(MODELE_DIR))
# model = AutoModelForSequenceClassification.from_pretrained(MODELE_DIR).eval()
#
# textes = dataset["test"].to_pandas()  # si la colonne texte a été conservée
# # Sinon, reconstruire les textes via src.data.clean_dataframe / split_dataframe.


### Grille d'analyse

Pour chaque erreur marquante, noter :

1. le texte (extrait) et l'étiquette réelle ;
2. la prédiction et la probabilité associée ;
3. l'hypothèse d'erreur : ironie, négation, texte trop long tronqué à 128 tokens,
   mélange de sentiments, vocabulaire hors domaine ;
4. une piste d'amélioration (max_length plus grand, plus de données, autre modèle).
